# T1 제출 코드 Colab 검증

**실제 GPU 실행 전 준비본입니다.** 위에서부터 실행합니다. 제출 ZIP의 코드를 인자 없이 `python script.py`로 실행하며 경로는 서버처럼 `PPS_*` 환경변수로 전달합니다.

1. NVIDIA GPU 런타임과 Colab 보안 비밀 `HF_TOKEN`을 준비합니다. 토큰과 모델 읽기 권한은 필수입니다.
2. 공개 저장소를 `git clone`하고 실제 커밋을 기록한 뒤 그 코드의 제출 ZIP을 만듭니다. 로컬 ZIP 업로드도 선택할 수 있습니다.
3. Python 3.12.13·고정 패키지를 별도 환경에 설치합니다.
4. 준비 단계에서 토큰으로 고정 리비전 모델을 다운로드하고 추론은 로컬 경로만 사용합니다.
5. 샘플 10건 → dev 200건 → 설정 검사·채점을 통과하면 **검증한 submit.zip 그대로 다운로드**합니다.
6. 실패하면 맨 아래 로그 다운로드 셀을 실행합니다.

큰 GPU에서도 실제 적재·속도를 확인해야 합니다. 같은 코드·모델·기본 설정을 검사하지만, 비공개 입력과 물리 GPU/메모리·서버 컨테이너 차이 때문에 서버 성공을 100% 보장할 수는 없습니다.

Colab 번들·결과 ZIP은 제출물이 아닙니다. 공개 샘플/dev만 사용합니다. 로컬 실행 안내: `docs/colab.md`.


In [1]:
import csv
import gzip
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

WORK = Path(tempfile.mkdtemp(prefix="t1-colab-", dir="/content"))
RESULTS = WORK / "results"
RESULTS.mkdir()
PYTHON = str(WORK / "venv/bin/python")
MODEL_ID = "google/gemma-4-26B-A4B-it"
REVISION = "4d7ae4984b7db7de8f8457170b3f1a419ee76d52"
EXPECTED_PACKAGES = {"vllm": "0.26.0", "torch": "2.11.0+cu130",
                     "transformers": "5.14.1", "xgrammar": "0.2.3"}
SERVER_PYTHON = "3.12.13"
case_inputs = {}
SOURCE_MODE = "clone"  # 특정 로컬 ZIP을 검사하려면 "upload"
REPO_URL = "https://github.com/LittleBitAI/ai-nara-shop.git"
REPO_REF = "9ed08054fc2c288a57fa33c689ea1f31703d15b2"  # 재현 실행에서는 이전 source.json의 commit SHA 지정 가능

def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n",
                    encoding="utf-8", newline="\n")

def run_logged(name, command, env=None, cwd=None):
    started = time.time()
    entry = {"argv": [str(x) for x in command], "cwd": str(cwd or WORK), "started": started,
             "returncode": None}
    if entry["argv"][0] == PYTHON:
        # Absolute Python paths do not activate PATH for tools such as ninja.
        env = dict(os.environ if env is None else env)
        env["PATH"] = str(Path(PYTHON).parent) + os.pathsep + env.get("PATH", "")
        env["VIRTUAL_ENV"] = str(Path(PYTHON).parent.parent)
    log_path = RESULTS / (name + ".log")
    record_path = RESULTS / (name + "-command.json")
    if log_path.exists() or record_path.exists():
        raise ValueError(f"{name} 실행 기록이 이미 있습니다. 첫 셀부터 새 작업 폴더로 실행하세요.")
    write_json(record_path, entry)
    try:
        with log_path.open("x", encoding="utf-8", newline="\n") as log_file:
            with subprocess.Popen(entry["argv"], cwd=entry["cwd"], env=env, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                  errors="replace", bufsize=1) as process:
                try:
                    for line in process.stdout:
                        log_file.write(line)
                        log_file.flush()
                        print(line, end="")
                    entry["returncode"] = process.wait()
                except BaseException:
                    process.terminate()
                    try:
                        process.wait(timeout=30)
                    except subprocess.TimeoutExpired:
                        process.kill()
                    raise
    finally:
        entry["elapsed_seconds"] = time.time() - started
        write_json(record_path, entry)
    if entry["returncode"] != 0:
        raise RuntimeError(f"{name} 실패 (exit={entry['returncode']}). 마지막 로그 다운로드 셀을 실행하세요.")
    return log_path

write_json(RESULTS / "host.json", {"python": sys.version, "work": str(WORK)})
print("작업 폴더:", WORK)


작업 폴더: /content/t1-colab-76jnzr39


## 1. git clone으로 코드·데이터와 제출 ZIP 준비

기본은 공개 저장소 clone입니다. 위 설정의 `REPO_REF`는 기본 `main`이며 실제 받은 커밋 SHA를 기록합니다.
재현할 때는 그 SHA를 지정하세요. clone한 코드의 기존 패키징 도구로 제출 ZIP과 검증 번들을 만듭니다.
아래 ZIP 검사는 자동 생성된 번들을 읽으며 수동 업로드가 필요 없습니다.

이미 만든 로컬 ZIP을 검사할 경우에만 첫 셀의 `SOURCE_MODE = "upload"`로 바꿉니다.
이때 로컬에서 만든 `colab-bundle.zip` 하나를 업로드합니다.


In [2]:
if SOURCE_MODE == "clone":
    REPO = WORK / "repo"
    BUNDLE_PATH = WORK / "candidate-colab-bundle.zip"
    run_logged("git-clone", ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO)])
    if REPO_REF != "main":
        # GitHub은 임의 SHA의 fetch를 40자 전체로만 받습니다. 약칭은 remote ref가 아니므로
        # `fatal: couldn't find remote ref`로 죽습니다. 브랜치 이름은 그대로 됩니다.
        if 7 <= len(REPO_REF) < 40 and all(c in "0123456789abcdef" for c in REPO_REF):
            raise ValueError(f'REPO_REF="{REPO_REF}"는 약칭 SHA입니다. 40자 전체 SHA나 브랜치 이름을 쓰세요.')
        run_logged("git-fetch-ref", ["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", REPO_REF])
        run_logged("git-checkout-ref", ["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"])
    commit_log = run_logged("git-commit", ["git", "-C", str(REPO), "rev-parse", "HEAD"])
    SOURCE_COMMIT = commit_log.read_text(encoding="utf-8").strip()
    write_json(RESULTS / "source.json", {"mode": "clone", "url": REPO_URL,
                                      "requested_ref": REPO_REF, "commit": SOURCE_COMMIT})
    run_logged("package", [sys.executable, "-X", "utf8", str(REPO / "tools/package.py"),
        "--output", str(WORK / "candidate-submit.zip"), "--colab-output", str(BUNDLE_PATH)], cwd=REPO)
    print("검증할 저장소 커밋:", SOURCE_COMMIT)
elif SOURCE_MODE != "upload":
    raise ValueError('SOURCE_MODE는 "clone" 또는 "upload"여야 합니다.')


Cloning into '/content/t1-colab-76jnzr39/repo'...
From https://github.com/LittleBitAI/ai-nara-shop
 * branch            9ed08054fc2c288a57fa33c689ea1f31703d15b2 -> FETCH_HEAD
HEAD is now at 9ed0805 run: B1 competitive_row 회차 코드 고정
9ed08054fc2c288a57fa33c689ea1f31703d15b2
{"path": "/content/t1-colab-76jnzr39/candidate-submit.zip", "bytes": 50465, "sha256": "f981c819f492d359dba307edc1488d3b200d8e38a1fc27b22a729463f4e1c1de"}
ZIP 검증 PASS (압축 해제 mock 10건·49열). 실제 모델·서버 제출 미검증.
Colab 전용 번들: /content/t1-colab-76jnzr39/candidate-colab-bundle.zip (대회 제출 금지)
검증할 저장소 커밋: 9ed08054fc2c288a57fa33c689ea1f31703d15b2


In [3]:
if SOURCE_MODE == "upload":
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("colab-bundle.zip 한 개만 선택하세요.")
    bundle_bytes = next(iter(uploaded.values()))
    del uploaded
    write_json(RESULTS / "source.json", {"mode": "upload"})
else:
    bundle_bytes = BUNDLE_PATH.read_bytes()
# tools/package.py의 COLAB_FILES와 같아야 합니다. 한쪽만 고치면 이 검사가 막습니다.
allowed = {"submit.zip", "tools/score.py", "tools/diagnose_items.py",
           "open/dev.jsonl", "open/dev_labels.csv",
           "open/data/test.jsonl.gz", "open/data/항목표.json", "open/data/정답스키마_디코딩.json",
           "open/data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법률.txt",
           "open/data/법령패키지/법령/중소기업제품 구매촉진 및 판로지원에 관한 법률 시행령.txt",
           "open/data/법령패키지/중기부고시/중기부고시_경쟁제품_세부품명.csv",
           "bundle-manifest.json"}
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    if set(archive.namelist()) != allowed or len(archive.namelist()) != len(allowed):
        raise ValueError("Colab 번들의 파일 목록이 다릅니다.")
    if sum(info.file_size for info in archive.infolist()) > 250_000_000:
        raise ValueError("Colab 번들이 예상 크기를 초과합니다.")
    manifest = json.loads(archive.read("bundle-manifest.json"))
    if set(manifest["sha256"]) != allowed - {"bundle-manifest.json"}:
        raise ValueError("번들 해시 목록이 다릅니다.")
    contents = {name: archive.read(name) for name in allowed}
    for name, expected in manifest["sha256"].items():
        if hashlib.sha256(contents[name]).hexdigest() != expected:
            raise ValueError(f"번들 해시 불일치: {name}")
    for name, content in contents.items():
        destination = WORK / name  # exact allowlist checked above
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(content)

SUBMISSION = WORK / "submission"
SUBMISSION.mkdir()
with zipfile.ZipFile(WORK / "submit.zip") as archive:
    if set(archive.namelist()) != {"script.py", "requirements.txt"} or len(archive.namelist()) != 2:
        raise ValueError("제출 ZIP 루트가 두 파일이 아닙니다.")
    if sum(info.file_size for info in archive.infolist()) > 10_000_000:
        raise ValueError("제출 ZIP이 예상 크기를 초과합니다.")
    for name in archive.namelist():
        (SUBMISSION / name).write_bytes(archive.read(name))
SCRIPT_SHA256 = hashlib.sha256((SUBMISSION / "script.py").read_bytes()).hexdigest()
write_json(RESULTS / "bundle-manifest.json", manifest)
write_json(RESULTS / "candidate.json", {
    "bundle_sha256": hashlib.sha256(bundle_bytes).hexdigest(),
    "submit_sha256": manifest["sha256"]["submit.zip"], "script_sha256": SCRIPT_SHA256,
    "model_id": MODEL_ID, "revision": REVISION,
})
print("제출 ZIP SHA-256:", manifest["sha256"]["submit.zip"])
print("실행 script.py SHA-256:", SCRIPT_SHA256)
del bundle_bytes, contents


제출 ZIP SHA-256: f981c819f492d359dba307edc1488d3b200d8e38a1fc27b22a729463f4e1c1de
실행 script.py SHA-256: 967f65e61c2b2bedbcc471a6be70ceabc89eaf6fb34eaa62b26186958c649419


## 2. 실제 GPU와 디스크 확인

GPU 종류·드라이버·VRAM·RAM을 기록합니다. 30GiB VRAM/80GiB 여유 디스크는 이 노트북의 사전 거름 기준이며 적재 보장이 아닙니다. 부족하면 모델 다운로드 전에 멈춥니다.


In [4]:
gpu_log = run_logged("gpu", ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                             "--format=csv,noheader"])
gpus = gpu_log.read_text(encoding="utf-8").strip().splitlines()
if not gpus or max(int(row.split(",")[1].strip().split()[0]) for row in gpus) < 30 * 1024:
    raise RuntimeError("VRAM 30GiB 미만입니다. 더 큰 GPU 런타임을 선택한 뒤 처음부터 실행하세요.")
resources = {"gpus": gpus, "disk_free_bytes": shutil.disk_usage(WORK).free,
             "meminfo": Path("/proc/meminfo").read_text()}
write_json(RESULTS / "resources.json", resources)
if resources["disk_free_bytes"] < 80 * 1024**3:
    raise RuntimeError("설치와 원본 모델을 준비할 여유 디스크 80GiB가 필요합니다.")
print("GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.")


NVIDIA A100-SXM4-40GB, 40960 MiB, 580.82.07
GPU 사전 확인 통과. 실제 적재/실행 성공은 아직 확인하지 않았습니다.


## 3. 대회 Python·패키지·설치 경로 재현

uv는 정확한 Python 3.12.13을 준비하는 Colab 도구로만 설치합니다. 추론 패키지는 별도 venv에 고정 버전으로 설치합니다.
이후 제출 ZIP의 requirements.txt를 설치하고, 실제 Python·핵심 패키지·CUDA 빌드가 명세와 다르면 중단합니다.

[대회 서버 명세](https://www.dacon.io/competitions/official/236754/overview/evaluation) · [uv Python 관리](https://docs.astral.sh/uv/guides/install-python/).
서버 컨테이너 이미지 전체나 호스트 GPU/드라이버까지 동일하게 복제하는 것은 아닙니다.


In [5]:
# Colab 커널 Python과 분리해 대회 서버의 정확한 Python 버전을 준비합니다.
run_logged("uv-install", [sys.executable, "-m", "pip", "install", "uv"])
run_logged("python-install", [sys.executable, "-m", "uv", "venv", "--python", SERVER_PYTHON,
                             "--seed", str(WORK / "venv")])
run_logged("install", [PYTHON, "-m", "pip", "install",
    "--extra-index-url", "https://download.pytorch.org/whl/cu130",
    *[f"{name}=={version}" for name, version in EXPECTED_PACKAGES.items()]])
# 서버처럼 제출 ZIP 안의 requirements.txt도 설치합니다.
run_logged("submission-install", [PYTHON, "-m", "pip", "install", "-r",
                                  str(SUBMISSION / "requirements.txt")])
if json.loads((RESULTS / "submission-install-command.json").read_text())["elapsed_seconds"] > 600:
    raise RuntimeError("제출 requirements 설치가 서버 제한 600초를 넘었습니다.")
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze"])
runtime_code = """
import json, platform, shutil, subprocess, sys, torch, vllm, transformers, xgrammar
from importlib.metadata import version
from pathlib import Path
runtime = {"python": platform.python_version(), "cuda": torch.version.cuda,
           "packages": {n: version(n) for n in ["torch", "vllm", "transformers", "xgrammar"]},
           "gpu": torch.cuda.get_device_name(0),
           "ninja_path": shutil.which("ninja"),
           "ninja_version": subprocess.check_output(["ninja", "--version"], text=True).strip()}
Path(sys.argv[1]).write_text(json.dumps(runtime, indent=2) + "\\n", encoding="utf-8")
print(runtime)
torch.empty(1, device="cuda")
"""
run_logged("runtime", [PYTHON, "-c", runtime_code, str(RESULTS / "runtime.json")])
runtime = json.loads((RESULTS / "runtime.json").read_text())
if (runtime["python"] != SERVER_PYTHON or runtime["packages"] != EXPECTED_PACKAGES
        or runtime["cuda"] != "13.0"):
    raise RuntimeError("서버의 Python/핵심 패키지/CUDA 빌드와 다릅니다. 검증을 중단합니다.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 89.7 MB/s eta 0:00:00
Using CPython 3.12.13
Creating virtual environment with seed packages at: venv
 + pip==26.2.1
Activate with: source venv/bin/activate
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 124.5 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 124.2 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 117.1 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 126.5 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 122.1 MB/s  0:00:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 208.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 124.8 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 230.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 150.4 

## 4. HF_TOKEN으로 고정 리비전 모델 다운로드

Colab 보안 비밀에 `HF_TOKEN`을 등록하고 이 노트북의 접근을 허용하세요. 읽지 못하면 숨김 입력으로 받으며 **빈 토큰은 거부**합니다.
토큰은 모델 다운로드 자식 프로세스 환경변수로만 전달하고, 추론 환경·명령행·로그·결과 ZIP에 기록하지 않습니다.

[고정 모델](https://huggingface.co/google/gemma-4-26B-A4B-it/tree/4d7ae4984b7db7de8f8457170b3f1a419ee76d52)의 접근 권한이 있는 계정을 사용합니다.
가중치는 준비 단계에서만 내려받고 제출물에 포함하지 않습니다.


In [6]:
from google.colab import userdata
from getpass import getpass

try:
    token = userdata.get("HF_TOKEN")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    token = getpass("HF_TOKEN (Hugging Face 읽기 토큰, 필수): ")
if not token or not token.strip():
    raise ValueError("HF_TOKEN이 필요합니다. Colab 보안 비밀에 등록하고 노트북 접근을 허용하세요.")
download_env = {**os.environ, "HF_TOKEN": token.strip()}
download_env.pop("HF_HUB_OFFLINE", None)
download_env.pop("TRANSFORMERS_OFFLINE", None)
download_code = """
from huggingface_hub import snapshot_download
from pathlib import Path
import os
import sys
model_dir = snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], token=os.environ["HF_TOKEN"],
    allow_patterns=["*.safetensors", "*.json", "*.jinja", "*.model"])
if Path(model_dir).name != sys.argv[2]:
    raise RuntimeError("고정 snapshot 경로 불일치")
Path(sys.argv[3]).write_text(model_dir, encoding="utf-8")
"""
try:
    run_logged("model-download", [PYTHON, "-c", download_code, MODEL_ID, REVISION,
                                 str(WORK / "model-path.txt")], env=download_env)
finally:
    download_env.pop("HF_TOKEN", None)
    token = None
MODEL_DIR = (WORK / "model-path.txt").read_text(encoding="utf-8")
write_json(RESULTS / "model.json", {"id": MODEL_ID, "revision": REVISION, "path": MODEL_DIR})



Fetching 9 files: 100%|██████████| 9/9 [13:12<00:00, 88.02s/it] 


## 5. 서버 진입점으로 샘플 10건 실행

제출 ZIP 코드를 별도 디렉터리에 놓고 `python script.py`로 실행합니다. 경로만 PPS 환경변수로 지정합니다.
양자화·문맥·출력 예산·청크·seed 등은 제출 코드 기본값을 쓰고, 성공 보고서의 실제 설정도 확인합니다.
`--debug-responses`·`--mock`·Colab 전용 추론 옵션을 사용하지 않습니다. 실패 시 마지막 로그 다운로드 셀로 이동합니다.


In [7]:
def run_case(name, source, args=()):
    case = WORK / "cases" / name
    data = case / "data"
    data.mkdir(parents=True)
    for filename in ("script.py", "requirements.txt"):
        shutil.copyfile(SUBMISSION / filename, case / filename)
    for filename in ("항목표.json", "정답스키마_디코딩.json"):
        shutil.copyfile(WORK / "open/data" / filename, data / filename)
    shutil.copytree(WORK / "open/data/법령패키지", data / "법령패키지")
    # dev도 서버와 같은 test.jsonl.gz 입력 경로를 거칩니다.
    if source.suffix == ".gz":
        shutil.copyfile(source, data / "test.jsonl.gz")
    else:
        # mtime=0으로 고정해야 같은 dev.jsonl이 회차마다 같은 .gz 바이트가 됩니다.
        # gzip 기본값은 현재 시각을 헤더에 넣어 input_sha256이 매번 달라집니다.
        with source.open("rb") as original, (data / "test.jsonl.gz").open("wb") as raw:
            with gzip.GzipFile(fileobj=raw, mode="wb", mtime=0) as compressed:
                shutil.copyfileobj(original, compressed)
    case_inputs[name] = hashlib.sha256((data / "test.jsonl.gz").read_bytes()).hexdigest()
    env = {key: value for key, value in os.environ.items()
           if not key.startswith(("PPS_", "VLLM_")) and key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
    env.update(PPS_MODEL_DIR=MODEL_DIR, PPS_DATA_DIR=str(data), PPS_OUTPUT_DIR=str(RESULTS / name),
               HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUNBUFFERED="1",
               CUDA_VISIBLE_DEVICES="0", VLLM_NO_USAGE_STATS="1", HF_HUB_DISABLE_TELEMETRY="1")
    # 검증 회차는 인자를 넘기지 않습니다. check_live가 무인자 실행을 다시 확인합니다.
    # args는 진단 전용 회차(--debug-responses)에만 씁니다. 그 회차는 검증 통과로 세지 않습니다.
    run_logged(name, [PYTHON, "script.py", *args], env=env, cwd=case)

def check_live(name, expected):
    report = json.loads((RESULTS / name / "run_report.json").read_text(encoding="utf-8"))
    if (report["mode"] != "live" or report["model_success_count"] != expected
            or report.get("sme_model_success_count") != report.get("sme_verified_count")
            or report.get("sme_verified_count", -1) < 0 or report.get("sme_fallback_count", -1) < 0
            or report.get("sme_verified_count", -1) + report.get("sme_fallback_count", -1) != report.get("sme_selected_count")
            or report.get("sme_skipped_count", -1) < 0
            or report.get("sme_selected_count", -1) + report.get("sme_skipped_count", -1) != expected
            or report["건수"] != expected or report["자가검증"] != "PASS"
            or report["code_sha256"] != SCRIPT_SHA256):
        raise RuntimeError("실제 모델 성공 건수/코드/CSV 검사 불일치")
    if report["sme_fallback_count"]:
        print(f"{name}: 추가 분석 실패 {report['sme_fallback_count']}건은 검증된 기본 모델 판정을 보존했습니다.")
    reproduction = report["reproduction"]
    settings = reproduction["settings"]
    expected_settings = {"quant": "int8_per_channel_weight_only", "max_model_len": 16384,
        "max_tokens": 2048, "max_chars": 16000, "chunk": 128, "gpu_mem": 0.92, "tp": 1,
        "seed": 20260826, "temperature": 0, "thinking": False, "limit": None, "debug_responses": False,
        "sme_items": ["v13"], "sme_selection": "baseline_v13_positive",
        "prompt_language": "en_with_ko_legal_terms", "sme_facts": True}
    if any(settings.get(key) != value for key, value in expected_settings.items()):
        raise RuntimeError("서버 기본 추론 설정 불일치")
    if (reproduction["python"] != SERVER_PYTHON
            or any(reproduction["packages"].get(key) != value for key, value in EXPECTED_PACKAGES.items())
            or report["environment"].get("cuda") != "13.0"):
        raise RuntimeError("서버 Python/패키지/CUDA 빌드 불일치")
    # 기록된 model_dir은 개인 절대 경로를 남기지 않으므로 고정 리비전 이름으로 대조합니다.
    if (report["model"] != {"id": MODEL_ID, "expected_revision": REVISION}
            or Path(settings["model_dir"]).name != REVISION or Path(MODEL_DIR).name != REVISION
            or report["input_sha256"] != case_inputs[name]):
        raise RuntimeError("모델 리비전/경로 또는 실행 입력 불일치")
    for path, expected_hash in manifest["sha256"].items():
        if path.startswith("open/data/법령패키지/"):
            if reproduction["asset_sha256"].get(path.removeprefix("open/data/")) != expected_hash:
                raise RuntimeError("실행에 사용한 법령/품목 파일 해시 불일치")
    command = json.loads((RESULTS / (name + "-command.json")).read_text())
    if command["returncode"] != 0 or command["argv"] != [PYTHON, "script.py"]:
        raise RuntimeError("서버 무인자 실행 명령/종료 코드 불일치")
    if command["elapsed_seconds"] > 7200:
        raise RuntimeError("실행이 서버 제한 7200초를 넘었습니다.")
    # 실행 후에도 내보낼 ZIP과 실제 실행 파일이 같은지 확인합니다.
    columns = ["id"] + [f"v{i}" for i in range(1, 25)] + [f"e{i}" for i in range(1, 25)]
    paired = []
    for filename in ("baseline_submission.csv", "submission.csv"):
        with (RESULTS / name / filename).open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != columns:
                raise RuntimeError("비교 CSV 헤더 불일치")
            paired.append(list(reader))
    if any(len(rows) != expected for rows in paired):
        raise RuntimeError("비교 CSV 건수 불일치")
    extra_owned = set()
    for phase_items in (settings.get("extra_call_items") or {}).values():
        extra_owned |= set(phase_items)
    owned = set(settings["sme_items"]) | extra_owned
    allowed = owned | {"e" + item[1:] for item in owned}
    protected = set(columns) - allowed
    if any(before[key] != after[key] for before, after in zip(*paired) for key in protected):
        raise RuntimeError("추가 호출이 대상 밖 항목/근거 또는 ID를 변경했습니다.")
    # 선택은 후처리 **전** 원판정의 v13 양성으로 한다(`baseline_v13_positive`).
    # `baseline_submission.csv`는 후처리를 거친 뒤이고, 근거 계약이 검증된 인용 없는 양성을
    # 내리므로 그 v13 양성은 선택 집합의 **부분집합**이다. 같기를 요구하면 계약이 한 건이라도
    # 일하는 순간 빨개진다 — dev 200건에서 실제로 104건 중 25건이 내려간다.
    # 지켜야 할 것은 포함 관계다. 살아남은 양성이 선택보다 많으면 추가 분석이 그 건을 못 봤다.
    if sum(row["v13"] == "1" for row in paired[0]) > report["sme_selected_count"]:
        raise RuntimeError("v13 추가 분석이 기본 판정 양성 일부를 선택하지 않았습니다")
    # SME 생략은 다른 단계의 변경 권한을 취소하지 않는다. A1도 v13을 소유한다.
    skipped_protected = {key for item in set(settings["sme_items"]) - extra_owned
                         for key in (item, "e" + item[1:])}
    if any(before[key] != after[key] for before, after in zip(*paired)
           if before["v13"] == "0" for key in skipped_protected):
        raise RuntimeError("추가 분석을 생략한 공고가 변경됐습니다.")
    with zipfile.ZipFile(WORK / "submit.zip") as archive:
        for filename in ("script.py", "requirements.txt"):
            if archive.read(filename) != (WORK / "cases" / name / filename).read_bytes():
                raise RuntimeError("검증 중 실행 파일이 제출 ZIP과 달라졌습니다.")
    return report

run_case("sample", WORK / "open/data/test.jsonl.gz")
sample_report = check_live("sample", 10)
print("샘플 10건: 실제 모델·서버 진입점·설정 검사 통과. 다음은 dev 전체 검사입니다.")


[baseline] 입력 10건 ← data/test.jsonl.gz
[baseline] vllm 0.26.0 · 모델 <외부>/4d7ae4984b7db7de8f8457170b3f1a419ee76d52 · quant=int8_per_channel_weight_only · max_model_len=16384
INFO 09-23 04:55:56 [api_utils.py:273] non-default args: {'tokenizer': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52', 'seed': 20260826, 'max_model_len': 16384, 'disable_log_stats': True, 'quantization': 'int8_per_channel_weight_only', 'quantization_config': QuantizationConfigArgs(linear=None, moe=QuantSpec(weight=QuantKey(dtype=torch.int8, scale=ScaleDesc(dtype=torch.float32, static=True, group_shape=GroupShape(row=-1, col=1)), scale2=None, symmetric=True), activation=None), ignore=[]), 'model': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52'}
INFO 09-23 04:56:09 [model.py:623] Resolved architecture: Gemma4ForConditionalGeneration
INFO 09-23 04:56:09 [model.py:1788] Using max mod

## 6. 서버 입력 경로로 dev 200건 실행

dev를 `data/test.jsonl.gz`로 준비하고 샘플과 똑같이 `python script.py`를 실행합니다.
모델·버전·기본 설정·공고 수·ZIP 바이트 일치와 종료 코드·실행 시간 제한을 검사합니다.
실제 평가 데이터 1,853건과 그 처리 시간은 이 200건 검사로 증명할 수 없습니다.


In [8]:
check_live("sample", 10)
run_case("dev", WORK / "open/dev.jsonl")
dev_report = check_live("dev", 200)
print("dev 200건: 실제 모델·서버 진입점·설정 검사 통과.")


[baseline] 입력 200건 ← data/test.jsonl.gz
[baseline] vllm 0.26.0 · 모델 <외부>/4d7ae4984b7db7de8f8457170b3f1a419ee76d52 · quant=int8_per_channel_weight_only · max_model_len=16384
INFO 09-23 05:04:27 [api_utils.py:273] non-default args: {'tokenizer': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52', 'seed': 20260826, 'max_model_len': 16384, 'disable_log_stats': True, 'quantization': 'int8_per_channel_weight_only', 'quantization_config': QuantizationConfigArgs(linear=None, moe=QuantSpec(weight=QuantKey(dtype=torch.int8, scale=ScaleDesc(dtype=torch.float32, static=True, group_shape=GroupShape(row=-1, col=1)), scale2=None, symmetric=True), activation=None), ignore=[]), 'model': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52'}
INFO 09-23 05:04:27 [model.py:623] Resolved architecture: Gemma4ForConditionalGeneration
INFO 09-23 05:04:27 [model.py:1788] Using max mo

## 7. 채점하고 검증한 제출 ZIP 다운로드

샘플·dev의 실제 모델 실행과 서버 계약 검사를 통과한 경우에만 채점·통과 기록을 만들고 `submit.zip`을 내려받습니다.
이 ZIP을 그대로 제출하세요. Colab 통과 뒤 코드를 변경하거나 ZIP을 다시 만들면 검증을 다시 수행해야 합니다.


In [9]:
check_live("sample", 10)
dev_report = check_live("dev", 200)
run_logged("score", [PYTHON, str(WORK / "tools/score.py"),
    "--truth", str(WORK / "open/dev_labels.csv"), "--pred", str(RESULTS / "dev/submission.csv"),
    "--output-dir", str(RESULTS / "score")])
metrics = json.loads((RESULTS / "score/metrics.json").read_text(encoding="utf-8"))
run_logged("score-baseline", [PYTHON, str(WORK / "tools/score.py"),
    "--truth", str(WORK / "open/dev_labels.csv"),
    "--pred", str(RESULTS / "dev/baseline_submission.csv"),
    "--output-dir", str(RESULTS / "score-baseline")])
paired_metrics = json.loads((RESULTS / "score-baseline/metrics.json").read_text(encoding="utf-8"))
# 사용자 Colab 1c64604 성공 실행의 동일 dev 기준선입니다. 제출 추론에는 사용하지 않습니다.
baseline_inputs = {"open/dev.jsonl": "5507f5ab0ba53b708f87211ed050dbe531b48a626ace6084c3ccaefb53147534",
                   "open/dev_labels.csv": "84c79302ac190b45b2487ec8e02aab73e59071ee745828e813a7db96d3a97a35"}
if any(hashlib.sha256((WORK / path).read_bytes()).hexdigest() != expected
       for path, expected in baseline_inputs.items()):
    raise RuntimeError("성능 비교 기준선과 dev 입력/정답이 다릅니다.")
baseline_f1 = 0.2208013652894021
comparison = {"baseline_commit": "1c646047ce2a37ca5b27d1b4c43a4b0a32ce6f89",
              "baseline_macro_f1": baseline_f1, "candidate_macro_f1": metrics["macro_f1"],
              "macro_f1_delta": metrics["macro_f1"] - baseline_f1,
              "paired_baseline_macro_f1": paired_metrics["macro_f1"],
              "paired_baseline_description": "English instructions, Korean legal terms, 24-item first pass",
              "paired_macro_f1_delta": metrics["macro_f1"] - paired_metrics["macro_f1"],
              "paired_baseline_items": {k: paired_metrics["items"][k] for k in ("v10", "v11", "v13")},
              "pilot_items": {k: metrics["items"][k] for k in ("v10", "v11", "v13")}}
quality_pass = metrics["macro_f1"] > paired_metrics["macro_f1"] and metrics["macro_f1"] >= baseline_f1
comparison["quality_pass"] = quality_pass
write_json(RESULTS / "quality-comparison.json", comparison)
print("기준선 대비 성능 비교 (실행 성공과 별도):", comparison)
submit_hash = hashlib.sha256((WORK / "submit.zip").read_bytes()).hexdigest()
if submit_hash != manifest["sha256"]["submit.zip"]:
    raise RuntimeError("제출 ZIP이 업로드 시점과 달라졌습니다.")
write_json(RESULTS / "validation.json", {
    "status": "colab_pass", "submit_sha256": submit_hash, "sample_count": 10, "dev_count": 200,
    "macro_f1": metrics["macro_f1"], "server_success_guaranteed": False,
    "quality_pass": quality_pass,
    "remaining_differences": ["비공개 평가 입력 1853건", "GPU/가용 메모리/CPU/RAM/OS",
                              "서버 전체 실행 시간", "컨테이너 digest와 OS 수준 네트워크 차단"],
})
print("Colab 검증 통과. 아래 ZIP이 실제 검증한 제출 후보입니다:", submit_hash)
# 재패키징 없이 검증한 ZIP을 그대로 내려받습니다.
from google.colab import files
if quality_pass:
    files.download(str(WORK / "submit.zip"))
else:
    print("점수 개선 기준 미달: 제출 ZIP 자동 다운로드를 보류합니다. 마지막 셀에서 로그를 내려받으세요.")


Macro F1=0.575482169261; errors=131; output=/content/t1-colab-76jnzr39/results/score
Macro F1=0.404648216993; errors=229; output=/content/t1-colab-76jnzr39/results/score-baseline
기준선 대비 성능 비교 (실행 성공과 별도): {'baseline_commit': '1c646047ce2a37ca5b27d1b4c43a4b0a32ce6f89', 'baseline_macro_f1': 0.2208013652894021, 'candidate_macro_f1': 0.575482169261194, 'macro_f1_delta': 0.3546808039717919, 'paired_baseline_macro_f1': 0.40464821699341824, 'paired_baseline_description': 'English instructions, Korean legal terms, 24-item first pass', 'paired_macro_f1_delta': 0.17083395226777576, 'paired_baseline_items': {'v10': {'tp': 0, 'fp': 0, 'fn': 7, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'support': 7}, 'v11': {'tp': 2, 'fp': 2, 'fn': 4, 'precision': 0.5, 'recall': 0.3333333333333333, 'f1': 0.4, 'support': 6}, 'v13': {'tp': 3, 'fp': 76, 'fn': 3, 'precision': 0.0379746835443038, 'recall': 0.5, 'f1': 0.07058823529411765, 'support': 6}}, 'pilot_items': {'v10': {'tp': 4, 'fp': 7, 'fn': 3, 'precision': 0

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. 원응답 보존 (기본 실행) · 항목 진단 (선택)

`--debug-responses`로 dev를 다시 실행해 원응답을 기본 보존합니다. 채택 후보는 같은 ZIP으로 한 번 더 실행해 그날 그 코드의 churn을 측정합니다.
**서버 진입점 검증 통과와는 별개입니다.** 원응답과 항목별 변화는 후보 비교에 쓰고, `colab_pass`·ZIP 다운로드 검사는 기본 회차에서 수행합니다.
`docs/colab.md`가 정한 대로 기본 검증 회차는 무인자 실행을 유지합니다. 원응답 회차는 별도로 기록하며 과거 churn 값을 새 코드의 허용선으로 쓰지 않습니다.
결과는 `results/dev-debug/`에 들어가 마지막 로그 셀의 ZIP에 함께 담깁니다.

항목 진단은 `tools/diagnose_items.py`로 별도 스키마 질의를 돌립니다. 제출물 `script.py`는
읽기만 하며 `SME_ITEMS`·`submission.csv`는 그대로입니다. 부재탐지 5항목은 제출 스키마가
`근거문구`를 null로 고정하므로 원응답으로는 안 보입니다. 이 질의가 그 자리를 대신합니다.
결과는 `results/diagnose/`의 `items.jsonl`·`manifest.json`입니다.


In [10]:
# 진단 전용 회차입니다. 검증 통과가 아니며 제출 후보를 만들지 않습니다.
RUN_DIAGNOSTIC = True         # 기본 원응답 보존. dev 200건 재실행이며 채택 후보는 같은 ZIP으로 재측정합니다.
DIAGNOSTIC_ARGS = ["--debug-responses"]

if RUN_DIAGNOSTIC:
    run_case("dev-debug", WORK / "open/dev.jsonl", args=DIAGNOSTIC_ARGS)
    report = json.loads((RESULTS / "dev-debug/run_report.json").read_text(encoding="utf-8"))
    settings = report["reproduction"]["settings"]
    if not settings["debug_responses"] or report["mode"] != "live":
        raise RuntimeError("진단 회차가 원응답을 켜지 않았습니다.")
    events = [json.loads(line) for line
              in (RESULTS / "dev-debug/diagnostics.jsonl").read_text(encoding="utf-8").splitlines()]
    responses = [event for event in events if event["event"] == "response"]
    with_text = [event for event in responses if "response_text" in event]
    if not responses or len(with_text) != len(responses):
        raise RuntimeError("원응답이 기록되지 않았습니다.")
    write_json(RESULTS / "diagnostic.json",
               {"case": "dev-debug", "purpose": "item_diagnosis_only", "colab_pass": False,
                "debug_responses": True, "responses": len(responses),
                "sme_items": settings["sme_items"],
                "note": "원응답·항목별 변화 비교용입니다. 서버 진입점 검증 통과를 대신하지 않습니다."})
    print(f"진단 회차 완료: 원응답 {len(with_text)}건을 dev-debug/diagnostics.jsonl에 남겼습니다.")
    print("v10·v11·v16·v18·v20은 스키마가 근거문구를 null로 고정하므로 원응답에 근거가 없습니다.")
    print("위반=0 항목의 인용은 postprocess가 버리므로, 원응답이 그 인용을 되살리는 유일한 기록입니다.")
else:
    print("진단 회차를 건너뜁니다. RUN_DIAGNOSTIC=True로 바꾸면 dev를 --debug-responses로 다시 실행합니다.")

# 항목 진단: 0점 항목이 어느 단계에서 막혔는지 모델에게 직접 묻습니다. 제출 판정을 바꾸지 않습니다.
DIAGNOSE_ITEMS = ""       # 예: "v16,v18,v20". 비우면 건너뜁니다.
DIAGNOSE_IDS = ""         # 예: "PPS-DEV-20,PPS-DEV-037". 비우면 dev 전체입니다.

if DIAGNOSE_ITEMS:
    env = {key: value for key, value in os.environ.items()
           if not key.startswith(("PPS_", "VLLM_")) and key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN")}
    env.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", PYTHONUNBUFFERED="1",
               CUDA_VISIBLE_DEVICES="0", VLLM_NO_USAGE_STATS="1", HF_HUB_DISABLE_TELEMETRY="1")
    run_logged("diagnose", [PYTHON, "-X", "utf8", str(WORK / "tools/diagnose_items.py"),
                            "--items", DIAGNOSE_ITEMS, "--ids", DIAGNOSE_IDS,
                            "--input", str(WORK / "open/dev.jsonl"),
                            "--data-dir", str(WORK / "open/data"),
                            "--labels", str(WORK / "open/dev_labels.csv"),
                            "--model-dir", MODEL_DIR,
                            "--script", str(SUBMISSION / "script.py"),
                            "--output-dir", str(RESULTS / "diagnose")], env=env, cwd=WORK)
    stages = json.loads((RESULTS / "diagnose/manifest.json").read_text(encoding="utf-8"))["stages"]
    print("항목별 막힌 단계:", json.dumps(stages, ensure_ascii=False))
    print("진단 질의 결과입니다. 제출 판정·점수가 아니며 submission.csv를 바꾸지 않았습니다.")
else:
    print("항목 진단을 건너뜁니다. DIAGNOSE_ITEMS에 항목을 넣으면 별도 질의를 돌립니다.")


[baseline] 입력 200건 ← data/test.jsonl.gz
[baseline] vllm 0.26.0 · 모델 <외부>/4d7ae4984b7db7de8f8457170b3f1a419ee76d52 · quant=int8_per_channel_weight_only · max_model_len=16384
INFO 09-23 05:19:50 [api_utils.py:273] non-default args: {'tokenizer': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52', 'seed': 20260826, 'max_model_len': 16384, 'disable_log_stats': True, 'quantization': 'int8_per_channel_weight_only', 'quantization_config': QuantizationConfigArgs(linear=None, moe=QuantSpec(weight=QuantKey(dtype=torch.int8, scale=ScaleDesc(dtype=torch.float32, static=True, group_shape=GroupShape(row=-1, col=1)), scale2=None, symmetric=True), activation=None), ignore=[]), 'model': '/root/.cache/huggingface/hub/models--google--gemma-4-26B-A4B-it/snapshots/4d7ae4984b7db7de8f8457170b3f1a419ee76d52'}
INFO 09-23 05:19:51 [model.py:623] Resolved architecture: Gemma4ForConditionalGeneration
INFO 09-23 05:19:51 [model.py:1788] Using max mo

## 8. 성공·실패와 관계없이 로그 다운로드

중간 셀이 실패했어도 이 셀을 실행합니다. stdout/stderr 전체·진단 JSONL·명령·환경·해시 및 성공 시 CSV·채점 결과를 모읍니다.
기본 실행은 원응답을 저장하지 않습니다. 원인 예외·종료 사유·토큰 수로 진단합니다.

서버 계약 검사 실패나 실제 모델 오류는 `validation.json` 통과로 표시하지 않습니다. Colab이 끊기기 전에 다운로드하세요.


In [11]:
from google.colab import files

archive_path = WORK / ("colab-results-" + str(time.time_ns()) + ".zip")
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RESULTS.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(RESULTS).as_posix())
print("검증 결과:", archive_path.name)
files.download(str(archive_path))


검증 결과: colab-results-1790141677344456786.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>